In [ ]:
# --- Instalación (ejecutar solo si es necesario) ---
# seaborn suele venir preinstalado en Colab, pero por si acaso:
# !pip install seaborn --quiet


# --- Imports ---
import pandas as pd
import numpy as np
import seaborn as sns


# Verifica las versiones instaladas en tu sesión de Colab
print(f'Pandas:  {pd.__version__}')   # anota para reproducibilidad
print(f'NumPy:   {np.__version__}')
print(f'Seaborn: {sns.__version__}')

In [ ]:
# Cargar Titanic desde seaborn (caché local o descarga automática)
import seaborn as sns

titanic = sns.load_dataset('titanic')
print(type(titanic))   # <class 'pandas.core.frame.DataFrame'>
print(titanic.shape)   # (891, 15)  → 891 pasajeros, 15 columnas

In [ ]:
# Forma estándar para cargar un CSV local o desde URL
from pathlib import Path


# Desde Drive (ruta local en Colab)
# ruta = Path('/content/drive/MyDrive/ML_con_sklearn/datos/titanic.csv')
# df = pd.read_csv(ruta)


# Desde URL directa (útil para reproducibilidad sin Drive)
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'


try:
    df_csv = pd.read_csv(url)
    print(f'CSV cargado: {df_csv.shape}')
except Exception as e:
    print(f'Error al cargar CSV: {e}')
    print('Usando dataset de seaborn como alternativa')
    df_csv = titanic.copy()

In [ ]:
# Pregunta 1: ¿Cómo lucen los primeros registros?
titanic.head()          # primeras 5 filas por defecto
# titanic.head(10)      # o las primeras N que quieras


# Pregunta 2: ¿Cuántos datos tengo y de qué tipo son?
titanic.info()
# Muestra: nombre de columna, tipo de dato, valores no nulos


# Pregunta 3: ¿Cuáles son las estadísticas básicas?
titanic.describe()      # solo columnas numéricas por defecto
# titanic.describe(include='all')  # incluye columnas categóricas


# Pregunta 4: ¿Cuántos valores faltantes hay?
faltantes = titanic.isnull().sum()
print(faltantes[faltantes > 0])  # solo las columnas con NaN

In [ ]:
# Inspeccionar una columna específica
print(titanic['age'].dtype)       # float64
print(titanic['age'].isnull().sum())  # 177 valores faltantes


# Contar valores únicos en una columna categórica
print(titanic['embarked'].value_counts())
# S    644
# C    168
# Q     77
# dtype: int64


# Proporciones en lugar de conteos
print(titanic['survived'].value_counts(normalize=True).round(2))
# 0    0.62
# 1    0.38

In [ ]:
# Selección por nombre (loc): usa etiquetas de filas y columnas
edad_clase = titanic.loc[:, ['age', 'pclass', 'survived']]
print(edad_clase.shape)  # (891, 3)

# Selección por posición (iloc): usa índices enteros
primeras_3 = titanic.iloc[:3, :4]   # filas 0-2, columnas 0-3

# Columna individual → devuelve una Series, no un DataFrame
edades = titanic['age']
print(type(edades))  # <class 'pandas.core.series.Series'>

# Columna individual como DataFrame (doble corchete)
edades_df = titanic[['age']]
print(type(edades_df))  # <class 'pandas.core.frame.DataFrame'>

In [ ]:
# Filtrado condicional: filas que cumplen una condición
sobrevivientes = titanic[titanic['survived'] == 1]
print(sobrevivientes.shape)  # (342, 15)

# Condición compuesta: '&' (and), '|' (or), '~' (not)
mujeres_1ra = titanic[(titanic['sex'] == 'female') & (titanic['pclass'] == 1)]
print(mujeres_1ra.shape)  # (94, 15)

# Separar features (X) de la variable objetivo (y)
columnas_num = ['age', 'fare', 'pclass', 'sibsp', 'parch']
X = titanic[columnas_num]
y = titanic['survived']
print('X shape:', X.shape)   # (891, 5)
print('y shape:', y.shape)   # (891,)

In [ ]:
# Estrategia 1: eliminar filas con NaN (drástico, solo si son pocas)
titanic_sin_nan = titanic.dropna(subset=['embarked'])
print(titanic_sin_nan.shape)  # (889, 15)  → solo se eliminaron 2 filas

# Estrategia 2: rellenar NaN con un valor fijo
titanic_copy = titanic.copy()
titanic_copy['embarked'] = titanic_copy['embarked'].fillna('S')  # moda

# Estrategia 3: rellenar NaN con la mediana (para columnas numéricas)
mediana_edad = titanic_copy['age'].median()
titanic_copy['age'] = titanic_copy['age'].fillna(mediana_edad)
print(titanic_copy['age'].isnull().sum())  # 0 → sin NaN

In [ ]:
# Ver los tipos actuales de todas las columnas
print(titanic_copy.dtypes)

# Convertir columna a tipo específico
titanic_copy['pclass'] = titanic_copy['pclass'].astype(int)

# Convertir columna categórica (ahora 'object') a tipo category
# Esto reduce memoria y habilita operaciones propias de categorías
titanic_copy['sex'] = titanic_copy['sex'].astype('category')
titanic_copy['embarked'] = titanic_copy['embarked'].astype('category')

# Ver cuánta memoria ahorramos
print(titanic_copy.memory_usage(deep=True).sum() / 1024, 'KB')

In [ ]:
# map: reemplaza valores en una Serie según un diccionario
titanic_copy['sexo_num'] = titanic_copy['sex'].map({'male': 0, 'female': 1})
print(titanic_copy[['sex', 'sexo_num']].head(3))
#    sex  sexo_num
# 0  male         0
# 1  female       1
# 2  female       1

# apply sobre una columna: aplica una función a cada elemento
titanic_copy['es_menor'] = titanic_copy['age'].apply(lambda x: 1 if x < 18 else 0)

# Suma de columnas: operación vectorizada (mucho más rápido que apply)
titanic_copy['familia'] = titanic_copy['sibsp'] + titanic_copy['parch']
print(titanic_copy['familia'].describe())

In [ ]:
# Seleccionar columnas finales para el modelo
columnas_modelo = ['pclass', 'sexo_num', 'age', 'fare', 'familia']

# Verificar que no queden NaN antes de convertir
assert titanic_copy[columnas_modelo].isnull().sum().sum() == 0, \
    'Aún hay NaN — revisar limpieza'

# Exportar a NumPy
X = titanic_copy[columnas_modelo].to_numpy()
y = titanic_copy['survived'].to_numpy()

print('X shape:', X.shape)    # (891, 5)
print('X dtype:', X.dtype)    # float64
print('y shape:', y.shape)    # (891,)

# Ya listo para scikit-learn
# from sklearn.linear_model import LogisticRegression
# model = LogisticRegression()
# model.fit(X, y)